# Module 17 lab · Getting hired

The code from every lesson of [this module](https://aicourses.dev/research/getting-hired), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 17.2 ML coding drills, timed and without AI

Causal multi-head attention, a training loop, sampling, the KV cache and LoRA, each written from memory with every tensor’s shape noted.

[Read the lesson](https://aicourses.dev/research/getting-hired/coding-drills)

### Attention and the training loop

**Drill 1: causal multi-head attention.** Project to queries, keys and values, split into heads, score, mask the future, softmax, mix the values, and join the heads back:

In [ ]:
import torch

def causal_mha(x, Wqkv, Wo, n_heads):
    B, T, C = x.shape                          # batch, time, channels
    hs = C // n_heads                          # head size
    q, k, v = (x @ Wqkv).split(C, dim=2)       # each [B, T, C]
    q, k, v = (t.view(B, T, n_heads, hs).transpose(1, 2)
               for t in (q, k, v))             # each [B, H, T, hs]
    att = q @ k.transpose(-2, -1) / hs ** 0.5  # [B, H, T, T]
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
    att = att.masked_fill(~mask, float("-inf")).softmax(-1)
    y = att @ v                                # [B, H, T, hs]
    y = y.transpose(1, 2).reshape(B, T, C)     # heads side by side
    return y @ Wo                              # [B, T, C]

torch.manual_seed(0)
x = torch.randn(2, 5, 8)                       # B=2, T=5, C=8
Wqkv, Wo = torch.randn(8, 24), torch.randn(8, 8)
out = causal_mha(x, Wqkv, Wo, n_heads=2)
x2 = x.clone()
x2[:, -1] += 1.0                         # change only the last token
same = torch.allclose(causal_mha(x2, Wqkv, Wo, 2)[:, :-1], out[:, :-1])
print(out.shape, same)                   # torch.Size([2, 5, 8]) True

**Drill 2: the training loop.** Data, forward, loss, zero the gradients, backward, step, and print the loss:

In [ ]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(0)
V, T, B = 10, 8, 32                       # vocab, length, batch
start = torch.randint(0, V, (512, 1))
data = (start + torch.arange(T + 1)) % V  # e.g. 3, 4, ... 9, 0, 1

model = torch.nn.Sequential(torch.nn.Embedding(V, 32),
                            torch.nn.Linear(32, V))
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
print(f"ln(V) = {math.log(V):.3f}")       # expected loss at the start

for step in range(301):
    idx = torch.randint(0, len(data), (B,))
    x, y = data[idx, :-1], data[idx, 1:]  # each [B, T]
    logits = model(x)                     # [B, T, V]
    loss = F.cross_entropy(logits.view(-1, V), y.reshape(-1))
    opt.zero_grad()                       # forget old gradients
    loss.backward()                       # new gradients
    opt.step()                            # update the weights
    if step % 100 == 0:
        print(step, round(loss.item(), 3))

### Sampling and the KV cache

**Drill 3: sampling with temperature and top-k.** Divide by the temperature, keep the top k, softmax, draw:

In [ ]:
import torch

def sample(logits, temperature=1.0, top_k=None, gen=None):
    logits = logits / temperature                 # [V]
    if top_k is not None:
        kth = torch.topk(logits, top_k).values[-1]  # k-th largest
        logits = logits.masked_fill(logits < kth, float("-inf"))
    probs = logits.softmax(-1)                    # [V], sums to 1
    return torch.multinomial(probs, 1, generator=gen).item()

logits = torch.tensor([2.0, 1.0, 0.5, -1.0])
gen = torch.Generator().manual_seed(0)
picks = [sample(logits, gen=gen) for _ in range(10000)]
freq = torch.bincount(torch.tensor(picks), minlength=4) / 10000
print(freq, logits.softmax(-1))                   # should be close
print({sample(logits, top_k=1, gen=gen) for _ in range(100)})  # {0}
print({sample(logits, top_k=2, gen=gen) for _ in range(100)})  # {0, 1}

**Drill 4: the KV cache.** When a model writes one token at a time, keep each token’s key and value instead of recomputing them. The check: the cached answer must equal full causal attention over all tokens at once.

In [ ]:
import torch

torch.manual_seed(0)
T, d = 6, 4                               # tokens, head size
X = torch.randn(T, d)                     # one head's input, [T, d]
Wq, Wk, Wv = (torch.randn(d, d) for _ in range(3))

# with a cache: one token at a time, keys and values kept
K_cache, V_cache, outs = [], [], []
for t in range(T):
    x = X[t]                              # the new token, [d]
    K_cache.append(x @ Wk)                # computed once, kept
    V_cache.append(x @ Wv)
    K, V = torch.stack(K_cache), torch.stack(V_cache)  # [t+1, d]
    w = (K @ (x @ Wq) / d ** 0.5).softmax(0)           # [t+1]
    outs.append(w @ V)                                 # [d]
cached = torch.stack(outs)                # [T, d]

# without a cache: all tokens at once, with a causal mask
Q, K, V = X @ Wq, X @ Wk, X @ Wv          # each [T, d]
future = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
att = (Q @ K.T / d ** 0.5).masked_fill(future, float("-inf"))
full = att.softmax(-1) @ V                # [T, d]
print(torch.allclose(cached, full, atol=1e-6))   # True

### LoRA

**Drill 5: a LoRA layer.** Freeze the original layer, add a low-rank update $BA$ scaled by $\alpha / r$, with B starting at zero. As in Module 13, $A$ and $B$ are the two thin matrices, $r$ is the rank and $\alpha$ is a setting for the update’s strength:

In [ ]:
import torch

class LoRALinear(torch.nn.Module):
    def __init__(self, base, r=4, alpha=8):
        super().__init__()
        self.base = base.requires_grad_(False)       # W stays frozen
        d_in, d_out = base.in_features, base.out_features
        self.A = torch.nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.B = torch.nn.Parameter(torch.zeros(d_out, r))  # zero
        self.scale = alpha / r

    def forward(self, x):                            # x [..., d_in]
        update = x @ self.A.T @ self.B.T             # [..., d_out]
        return self.base(x) + self.scale * update

torch.manual_seed(0)
base = torch.nn.Linear(512, 512)
lora = LoRALinear(base, r=4)
x = torch.randn(3, 512)
print(torch.equal(lora(x), base(x)))                 # True at the start
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(trainable, base.weight.numel())                # 4096 262144

## 17.3 Debugging and ML system design rounds

“Find the bug” katas from Module 8, and “how would you train a 100B model?” answered with Module 11 arithmetic.

[Read the lesson](https://aicourses.dev/research/getting-hired/debugging-and-system-design)

### Debugging: read the symptom, then the code

Here is a kata, a short drill to repeat, with two planted bugs, switched on one at a time:

In [ ]:
import torch
import torch.nn.functional as F

def train(bug=None, steps=301, V=10, T=8):
    torch.manual_seed(0)
    start = torch.randint(0, V, (512, 1))
    data = (start + torch.arange(T + 1)) % V       # counting, mod 10
    model = torch.nn.Sequential(torch.nn.Embedding(V, 32),
                                torch.nn.Linear(32, V))
    opt = torch.optim.SGD(model.parameters(), lr=0.5)
    for step in range(steps):
        idx = torch.randint(0, len(data), (32,))
        x, y = data[idx, :-1], data[idx, 1:]
        if bug == "no shift":
            y = x                                  # targets = inputs
        logits = model(x)                          # [B, T, V]
        if bug == "double softmax":
            logits = logits.softmax(-1)            # softmax twice
        loss = F.cross_entropy(logits.view(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    after_3 = model(torch.tensor([[3]])).argmax(-1).item()
    return round(loss.item(), 3), after_3

for bug in [None, "double softmax", "no shift"]:
    print(bug, train(bug))

### ML system design: arithmetic out loud

In [ ]:
N, D = 100e9, 2e12            # parameters, tokens: 20 per parameter
flops = 6 * N * D             # Module 11's rule
peak = 989e12                 # one H100, dense BF16, FLOP per second
mfu = 0.40                    # the share of peak a good run achieves
gpus = 1024

gpu_seconds = flops / (peak * mfu)
print(f"{flops:.1e} FLOPs")
print(f"{gpu_seconds / 3600:,.0f} GPU-hours")
print(f"{gpu_seconds / gpus / 86400:.0f} days on {gpus} GPUs")

state = 18 * N                # bytes: weights 6, Adam 8, gradients 4
print(f"{state / 1e12:.1f} TB of weights, gradients and Adam states")
print(f"{state / 80e9:.1f} GPUs of 80 GB, before any activations")
print(f"check against Llama 3 405B: {6 * 405e9 * 15.6e12:.2e}")